In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# SAE-SPECIFIC FEATURE ENGINEERING
# ALL SEPSIS ICU STAYS
# ============================================================

ROOT = Path("/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning")
MIMIC = ROOT / "data" / "raw" / "mimic-iv"

HOSP = MIMIC / "hosp"
ICU = MIMIC / "icu"

DATA = ROOT / "sae_mimiciv_v2" / "data"
DATA.mkdir(parents=True, exist_ok=True)

BASE = pd.read_csv(
    DATA / "sepsis_all_hourly_base.csv"
)

BASE["hour_time"] = pd.to_datetime(
    BASE["hour_time"]
)

print("Base dataset:", BASE.shape)
print("Patients:", BASE["subject_id"].nunique())
print("ICU stays:", BASE["stay_id"].nunique())
print("SAE rows:", BASE["sae"].sum())

# ============================================================
# 1. LOAD ICU ITEM DICTIONARY
# ============================================================

d_items = pd.read_csv(
    ICU / "d_items.csv.gz"
)

# More specific patterns than before
patterns = {
    "heart_rate": r"^Heart Rate$",
    "resp_rate": r"^Respiratory Rate$",
    "spo2": r"^O2 saturation pulseoxymetry$|^SpO2$",
    "map": r"mean arterial pressure",
    "temperature": r"temperature",
    "gcs_eye": r"gcs.*eye",
    "gcs_motor": r"gcs.*motor",
    "gcs_verbal": r"gcs.*verbal",
    "gcs_total": r"gcs.*total",
    "rass": r"richmond agitation|rass",
}

item_groups = {}

for feature, pattern in patterns.items():

    mask = d_items["label"].astype(str).str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )

    matches = d_items.loc[mask]

    item_groups[feature] = set(
        matches["itemid"].astype(int)
    )

    print(
        f"{feature:15s}: {len(matches):3d} items"
    )

# ============================================================
# 2. LOAD CHART EVENTS ONLY FOR OUR 26 ICU STAYS
# ============================================================

stay_ids = set(
    BASE["stay_id"].unique()
)

needed_items = set()

for ids in item_groups.values():
    needed_items.update(ids)

chart_parts = []

for chunk in pd.read_csv(
    ICU / "chartevents.csv.gz",
    compression="gzip",
    chunksize=500_000
):

    x = chunk[
        chunk["stay_id"].isin(stay_ids)
        &
        chunk["itemid"].isin(needed_items)
    ].copy()

    if len(x):
        chart_parts.append(x)

if chart_parts:
    chart = pd.concat(
        chart_parts,
        ignore_index=True
    )
else:
    chart = pd.DataFrame()

print("\nFiltered chart events:", chart.shape)

# ============================================================
# 3. CONVERT CHART EVENTS TO HOURLY FEATURES
# ============================================================

if len(chart):

    item_to_feature = {}

    for feature, ids in item_groups.items():
        for itemid in ids:
            item_to_feature[itemid] = feature

    chart["feature"] = chart["itemid"].map(
        item_to_feature
    )

    chart["charttime"] = pd.to_datetime(
        chart["charttime"]
    )

    chart["valuenum"] = pd.to_numeric(
        chart["valuenum"],
        errors="coerce"
    )

    chart = chart.dropna(
        subset=["valuenum", "charttime"]
    )

    chart["hour_time"] = (
        chart["charttime"].dt.floor("h")
    )

    chart_hourly = (
        chart
        .groupby(
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "hour_time",
                "feature"
            ],
            as_index=False
        )["valuenum"]
        .median()
    )

    chart_hourly = chart_hourly.pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="first"
    ).reset_index()

else:

    chart_hourly = pd.DataFrame()

print(
    "Hourly chart table:",
    chart_hourly.shape
)

# ============================================================
# 4. LOAD LAB DICTIONARY
# ============================================================

d_labitems = pd.read_csv(
    HOSP / "d_labitems.csv.gz"
)

lab_patterns = {
    "lactate": r"lactate",
    "wbc": r"white blood cell|wbc",
    "creatinine": r"creatinine",
    "bilirubin": r"bilirubin",
    "platelets": r"platelet",
    "bun": r"\bbun\b|urea nitrogen",
    "sodium": r"sodium",
    "glucose": r"glucose",
    "ph": r"blood ph|arterial ph|^ph$",
}

lab_groups = {}

for feature, pattern in lab_patterns.items():

    mask = d_labitems["label"].astype(str).str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )

    matches = d_labitems.loc[mask]

    lab_groups[feature] = set(
        matches["itemid"].astype(int)
    )

    print(
        f"{feature:15s}: {len(matches):3d} items"
    )

# ============================================================
# 5. LOAD LAB EVENTS FOR OUR SEPSIS ADMISSIONS
# ============================================================

hadm_ids = set(
    BASE["hadm_id"].unique()
)

needed_lab_items = set()

for ids in lab_groups.values():
    needed_lab_items.update(ids)

lab_parts = []

for chunk in pd.read_csv(
    HOSP / "labevents.csv.gz",
    compression="gzip",
    chunksize=500_000
):

    x = chunk[
        chunk["hadm_id"].isin(hadm_ids)
        &
        chunk["itemid"].isin(needed_lab_items)
    ].copy()

    if len(x):
        lab_parts.append(x)

if lab_parts:
    labs = pd.concat(
        lab_parts,
        ignore_index=True
    )
else:
    labs = pd.DataFrame()

print(
    "\nFiltered laboratory events:",
    labs.shape
)

# ============================================================
# 6. CONVERT LABS TO HOURLY FEATURES
# ============================================================

if len(labs):

    lab_to_feature = {}

    for feature, ids in lab_groups.items():
        for itemid in ids:
            lab_to_feature[itemid] = feature

    labs["feature"] = labs["itemid"].map(
        lab_to_feature
    )

    labs["charttime"] = pd.to_datetime(
        labs["charttime"]
    )

    labs["valuenum"] = pd.to_numeric(
        labs["valuenum"],
        errors="coerce"
    )

    labs = labs.dropna(
        subset=["valuenum", "charttime"]
    )

    # Connect labs to ICU stays
    stay_info = BASE[
        [
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ]
    ].drop_duplicates()

    labs = labs.merge(
        stay_info[
            [
                "subject_id",
                "hadm_id",
                "stay_id"
            ]
        ].drop_duplicates(),
        on=["subject_id", "hadm_id"],
        how="inner"
    )

    labs["hour_time"] = (
        labs["charttime"].dt.floor("h")
    )

    lab_hourly = (
        labs
        .groupby(
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "hour_time",
                "feature"
            ],
            as_index=False
        )["valuenum"]
        .median()
    )

    lab_hourly = lab_hourly.pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="first"
    ).reset_index()

else:

    lab_hourly = pd.DataFrame()

print(
    "Hourly laboratory table:",
    lab_hourly.shape
)

# ============================================================
# 7. MERGE FEATURES WITH ALL SEPSIS HOURS
# ============================================================

df = BASE.copy()

if len(chart_hourly):

    df = df.merge(
        chart_hourly,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        how="left"
    )

if len(lab_hourly):

    df = df.merge(
        lab_hourly,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        how="left"
    )

# ============================================================
# 8. MISSINGNESS INDICATORS
# ============================================================

protected = {
    "subject_id",
    "hadm_id",
    "stay_id",
    "hour_time",
    "hour",
    "sae"
}

clinical_features = [
    c for c in df.columns
    if c not in protected
]

for c in clinical_features:

    if pd.api.types.is_numeric_dtype(df[c]):

        df[c + "_missing"] = (
            df[c].isna().astype(int)
        )

# ============================================================
# 9. FORWARD-FILL WITHIN EACH ICU STAY
# ============================================================

df = df.sort_values(
    ["stay_id", "hour_time"]
)

numeric_features = [
    c for c in clinical_features
    if c in df.columns
    and pd.api.types.is_numeric_dtype(df[c])
]

df[numeric_features] = (
    df
    .groupby("stay_id")[numeric_features]
    .ffill()
)

# ============================================================
# 10. MEDIAN FALLBACK
# ============================================================

for c in numeric_features:

    median = df[c].median()

    if pd.notna(median):
        df[c] = df[c].fillna(median)

# ============================================================
# 11. TEMPORAL CHANGE FEATURES
# ============================================================

delta_candidates = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "temperature",
    "gcs_total",
    "gcs_eye",
    "gcs_motor",
    "gcs_verbal",
    "rass",
    "lactate",
    "wbc",
    "creatinine",
    "bilirubin",
    "platelets",
    "bun",
    "sodium",
    "glucose",
    "ph"
]

for c in delta_candidates:

    if c in df.columns:

        df[c + "_delta"] = (
            df
            .groupby("stay_id")[c]
            .diff()
        )

# ============================================================
# 12. 3-HOUR ROLLING FEATURES
# ============================================================

rolling_candidates = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "temperature",
    "gcs_total",
    "lactate",
    "wbc",
    "creatinine",
    "sodium",
    "glucose"
]

for c in rolling_candidates:

    if c in df.columns:

        df[c + "_mean3h"] = (
            df
            .groupby("stay_id")[c]
            .transform(
                lambda x:
                x.rolling(
                    3,
                    min_periods=1
                ).mean()
            )
        )

# ============================================================
# 13. SAVE FINAL FEATURE TABLE
# ============================================================

output = (
    DATA / "sepsis_sae_multimodal_hourly.csv"
)

df.to_csv(
    output,
    index=False
)

print("\n" + "=" * 60)
print("MULTIMODAL SAE FEATURE TABLE COMPLETE")
print("=" * 60)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Patients:", df.subject_id.nunique())
print("ICU stays:", df.stay_id.nunique())

print("\nSAE distribution:")
print(df["sae"].value_counts())

print("\nClinical features:")
print(
    [
        c for c in df.columns
        if c not in [
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time",
            "hour",
            "sae"
        ]
    ]
)

print("\nRemaining missing values:")
print(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .head(30)
)

print("\nSaved:")
print(output)

Base dataset: (3769, 6)
Patients: 17
ICU stays: 26
SAE rows: 1517
heart_rate     :   1 items
resp_rate      :   1 items
spo2           :   1 items
map            :   0 items
temperature    :   9 items
gcs_eye        :   3 items
gcs_motor      :   3 items
gcs_verbal     :   4 items
gcs_total      :   0 items
rass           :   1 items

Filtered chart events: (17942, 11)
Hourly chart table: (3714, 11)
lactate        :   8 items
wbc            :  11 items
creatinine     :  20 items
bilirubin      :  16 items
platelets      :   8 items
bun            :  10 items
sodium         :  13 items
glucose        :  13 items
ph             :   6 items

Filtered laboratory events: (3637, 16)
Hourly laboratory table: (1069, 13)

MULTIMODAL SAE FEATURE TABLE COMPLETE
Rows: 3769
Columns: 63
Patients: 17
ICU stays: 26

SAE distribution:
sae
0    2252
1    1517
Name: count, dtype: int64

Clinical features:
['gcs_eye', 'gcs_motor', 'gcs_verbal', 'heart_rate', 'resp_rate', 'spo2', 'temperature', 'bilirubin'